# CBR con vectorización numérica de casos y recuperación por similitud

Este notebook hace cuatro cosas:

1. **Carga la base de casos** ya construida 
2. **Convierte cada caso en valores numéricos** por bloques:
   - perfil del usuario
   - preferencias
   - **objetivos XAI** (`main_goals_raw`)
   - contexto de la imagen
3. **Calcula similitud** entre casos usando una combinación ponderada de bloques.
4. **Recupera vecinos similares** y propone una **recomendación de explicación** a partir de esos vecinos.


## 1. Imports y rutas

In [18]:

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim
from skimage.transform import resize

cwd = Path.cwd()
if (cwd / "cbr_case_base_outputs").exists():
    BASE_DIR = cwd
elif (cwd / "base_de_casos" / "cbr_case_base_outputs").exists():
    BASE_DIR = cwd / "base_de_casos"
else:
    raise FileNotFoundError("No encuentro la carpeta cbr_case_base_outputs desde el directorio actual")

OUTPUT_DIR = BASE_DIR / "cbr_similarity_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
IMAGE_METADATA_PATH = BASE_DIR / "cbr_case_base_outputs" / "image_metadata_template.csv"
IMAGE_MAP_PATH = BASE_DIR / "cbr_case_base_outputs" / "image_id_to_description_case_map.csv"
DESCRIPTIONS_PATH = BASE_DIR.parent / "generacion_descripcion_XAI" / "resultados_descripciones_xai" / "descripciones_por_imagen_xai.csv"

CANDIDATE_INPUTS = [
    BASE_DIR / "cbr_case_base_outputs" / "case_base_double_full.csv",
    BASE_DIR / "cbr_case_base_outputs" / "case_base_double_partial.csv",
]

input_path = None
for p in CANDIDATE_INPUTS:
    if p.exists():
        input_path = p
        break

if input_path is None:
    raise FileNotFoundError("No encuentro case_base_double_full.csv ni case_base_double_partial.csv")

df = pd.read_csv(input_path)
print("Archivo cargado:", input_path)
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])
df.head(3)


Archivo cargado: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/base_de_casos/cbr_case_base_outputs/case_base_double_full.csv
Filas: 297 | Columnas: 42


,case_id,user_id,image_id,image_label,selected_option,satisfaction,confidence,understanding,mean_helpfulness,age_range,...,model_predicted_class,model_confidence,initial_description,vqa_support_description,option_A_type,option_B_type,option_C_type,option_D_type,option_E_type,option_F_type
0,C0001,U001,1,img_01,Opción B,4,5,4,4.333333,18–24,...,Aguila,NaN,Ave rapaz posada en guante junto a una persona,Original image used as the reference for compa...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN
1,C0002,U001,2,img_02,Opción A,4,5,4,4.333333,18–24,...,Pato,NaN,A close-up profile of a duck showcases its vib...,The base image shows: A close-up profile of a ...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN
2,C0003,U001,3,img_03,Opción E,4,3,4,3.666667,18–24,...,Serpiente,NaN,A vibrant green snake with black stripes curls...,The base image shows: A vibrant green snake wi...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN


## 2. Funciones auxiliares de limpieza y codificación

In [19]:

# ---------- Utilidades base ----------

def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def minmax_series(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    s_filled = s.fillna(s.median())
    smin, smax = s_filled.min(), s_filled.max()
    if smax == smin:
        return pd.Series(np.zeros(len(s_filled)), index=s.index)
    return (s_filled - smin) / (smax - smin)

def align_columns(df_block, template_columns):
    return df_block.reindex(columns=template_columns, fill_value=0)

def safe_cosine_matrix(X):
    if X.shape[1] == 0:
        return np.zeros((X.shape[0], X.shape[0]))
    return cosine_similarity(X)

def safe_cosine_query_vs_base(query_X, base_X):
    if base_X.shape[1] == 0:
        return np.zeros(base_X.shape[0])
    return cosine_similarity(query_X, base_X).ravel()


# ---------- Ontologías / diccionarios ----------

AGE_MAP = {
    "18–24": 0,
    "18-24": 0,
    "25–34": 1,
    "25-34": 1,
    "35–44": 2,
    "35-44": 2,
    "45–54": 3,
    "45-54": 3,
    "55–64": 4,
    "55-64": 4,
    "65 o más": 5,
    "65 o mas": 5,
}

EDUCATION_MAP = {
    "bachillerato/fp": 0,
    "grado": 1,
    "master": 2,
    "máster": 2,
    "doctorado": 3,
    "prefiero no contestar": np.nan,
}

RESPONSE_LENGTH_MAP = {
    "corta: solo lo esencial": 0,
    "media: explicación breve con algo de detalle": 1,
    "larga: explicación completa y detallada": 2,
}

TECH_LEVEL_MAP = {
    "simple: lenguaje claro y sin tecnicismos": 0,
    "intermedio: algunos términos técnicos, pero fáciles de seguir": 1,
    "técnico: explicación más especializada y precisa": 2,
    "tecnico: explicación más especializada y precisa": 2,
}

ERROR_IMPACT_MAP = {
    "bajo: el error apenas tendría consecuencias": 0,
    "medio: el error podría causar cierta confusión o problema": 1,
    "alto: el error podría tener consecuencias importantes": 2,
}

OCCUPATION_PATTERNS = {
    "ocup_estudiante": ["estudiante"],
    "ocup_investigacion": ["investigador", "investigadora"],
    "ocup_docencia": ["docente", "profesor", "profesora"],
    "ocup_dev": ["desarrollador", "desarrolladora"],
    "ocup_prof_tech": ["sector tecnológico", "sector tecnologico", "/ ia / datos", "ia / datos"],
    "ocup_prof_otro": ["profesional de otro sector"],
    "ocup_prefiere_no": ["prefiero no contestar"],
}

EXPLANATION_TYPE_PATTERNS = {
    "pref_ejemplos": ["ejemplos similares"],
    "pref_contraejemplos": ["comparaciones o contraejemplos"],
    "pref_atributos": ["atributos o características", "atributos o caracteristicas"],
    "pref_reglas": ["reglas o razonamientos paso a paso"],
    "pref_visual": ["zonas destacadas", "visuales"],
    "pref_sin_preferencia": ["no tengo preferencia", "me da igual"],
}

XAI_GOAL_PATTERNS = {
    "goal_transparencia": ["transparencia"],
    "goal_eficiencia": ["eficiencia"],
    "goal_efectividad": ["efectividad"],
    "goal_confianza": ["confianza"],
    "goal_persuasion": ["persuasión", "persuasion"],
    "goal_satisfaccion": ["satisfacción", "satisfaccion"],
    "goal_educacion": ["educación", "educacion"],
    "goal_debugging": ["detección de errores", "deteccion de errores", "debugging"],
    "goal_escrutinio": ["escrutinio"],
}

def map_with_fallback(series, mapping):
    return series.apply(lambda x: mapping.get(normalize_text(x), np.nan))

def extract_flags_from_text(raw_text, patterns_dict):
    txt = normalize_text(raw_text)
    result = {}
    for label, patterns in patterns_dict.items():
        result[label] = int(any(p in txt for p in patterns))
    return result


## 3. Ingeniería de atributos

Aquí construimos los bloques numéricos.  
Cada bloque representa una parte de la **descripción del caso**:

- **perfil**
- **preferencias**
- **objetivos XAI**
- **imagen**


In [20]:

def prepare_case_dataframe(df_input):
    df = df_input.copy()

    # Asegurar columnas esperadas
    for col in [
        "main_goals_raw",
        "preferred_explanation_types_raw",
        "occupation_raw",
        "preferred_format",
        "preferred_response_length",
        "preferred_technical_level",
        "perceived_error_impact",
        "image_id",
        "image_label",
        "age_range",
        "education_level",
        "ai_knowledge_level",
        "domain_knowledge_level",
    ]:
        if col not in df.columns:
            df[col] = np.nan

    # ---- Bloque perfil ----
    profile = pd.DataFrame(index=df.index)
    profile["age_code"] = map_with_fallback(df["age_range"], AGE_MAP)
    profile["education_code"] = map_with_fallback(df["education_level"], EDUCATION_MAP)
    profile["ai_knowledge"] = pd.to_numeric(df["ai_knowledge_level"], errors="coerce")
    profile["domain_knowledge"] = pd.to_numeric(df["domain_knowledge_level"], errors="coerce")

    profile["age_code"] = minmax_series(profile["age_code"])
    profile["education_code"] = minmax_series(profile["education_code"])
    profile["ai_knowledge"] = minmax_series(profile["ai_knowledge"])
    profile["domain_knowledge"] = minmax_series(profile["domain_knowledge"])

    occupation_flags = df["occupation_raw"].fillna("").apply(
        lambda x: pd.Series(extract_flags_from_text(x, OCCUPATION_PATTERNS))
    )
    profile = pd.concat([profile, occupation_flags], axis=1).fillna(0)

    # ---- Bloque preferencias ----
    preferences = pd.DataFrame(index=df.index)
    preferences["response_length_code"] = map_with_fallback(df["preferred_response_length"], RESPONSE_LENGTH_MAP)
    preferences["technical_level_code"] = map_with_fallback(df["preferred_technical_level"], TECH_LEVEL_MAP)
    preferences["error_impact_code"] = map_with_fallback(df["perceived_error_impact"], ERROR_IMPACT_MAP)

    preferences["response_length_code"] = minmax_series(preferences["response_length_code"])
    preferences["technical_level_code"] = minmax_series(preferences["technical_level_code"])
    preferences["error_impact_code"] = minmax_series(preferences["error_impact_code"])

    # one-hot formato preferido
    fmt = pd.get_dummies(df["preferred_format"].fillna("desconocido"), prefix="fmt")
    preferences = pd.concat([preferences, fmt], axis=1)

    # multi-hot tipos de explicación preferidos
    expl_flags = df["preferred_explanation_types_raw"].fillna("").apply(
        lambda x: pd.Series(extract_flags_from_text(x, EXPLANATION_TYPE_PATTERNS))
    )
    preferences = pd.concat([preferences, expl_flags], axis=1).fillna(0)

    # ---- Bloque objetivos XAI ----
    xai_goals = df["main_goals_raw"].fillna("").apply(
        lambda x: pd.Series(extract_flags_from_text(x, XAI_GOAL_PATTERNS))
    ).fillna(0)

    # ---- Bloque imagen ----
    image_block = pd.get_dummies(df["image_id"].astype("Int64").astype(str), prefix="img")

    # Si hay metadatos de imagen, se añaden
    if "domain" in df.columns and df["domain"].notna().sum() > 0:
        image_block = pd.concat([image_block, pd.get_dummies(df["domain"].fillna("desconocido"), prefix="domain")], axis=1)
    if "model_predicted_class" in df.columns and df["model_predicted_class"].notna().sum() > 0:
        image_block = pd.concat([image_block, pd.get_dummies(df["model_predicted_class"].fillna("desconocido"), prefix="pred")], axis=1)
    if "model_confidence" in df.columns and pd.to_numeric(df["model_confidence"], errors="coerce").notna().sum() > 0:
        image_block["model_confidence_norm"] = minmax_series(pd.to_numeric(df["model_confidence"], errors="coerce"))

    # Relleno final
    profile = profile.fillna(0).astype(float)
    preferences = preferences.fillna(0).astype(float)
    xai_goals = xai_goals.fillna(0).astype(float)
    image_block = image_block.fillna(0).astype(float)

    return {
        "profile": profile,
        "preferences": preferences,
        "xai_goals": xai_goals,
        "image": image_block,
    }

blocks = prepare_case_dataframe(df)
for name, block in blocks.items():
    print(name, block.shape)
    display(block.head(2))


profile (297, 11)


,age_code,education_code,ai_knowledge,domain_knowledge,ocup_estudiante,ocup_investigacion,ocup_docencia,ocup_dev,ocup_prof_tech,ocup_prof_otro,ocup_prefiere_no
0,0.0,0.333333,0.75,0.5,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.333333,0.75,0.5,1.0,1.0,0.0,0.0,0.0,0.0,0.0


preferences (297, 13)


,response_length_code,technical_level_code,error_impact_code,fmt_Me da igual,fmt_Principalmente visual,fmt_Texto breve + imagen,fmt_Texto detallado + imagen,pref_ejemplos,pref_contraejemplos,pref_atributos,pref_reglas,pref_visual,pref_sin_preferencia
0,0.5,0.5,0.5,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,0.5,0.5,0.5,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


xai_goals (297, 9)


,goal_transparencia,goal_eficiencia,goal_efectividad,goal_confianza,goal_persuasion,goal_satisfaccion,goal_educacion,goal_debugging,goal_escrutinio
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0


image (297, 21)


,img_1,img_10,img_11,img_2,img_3,img_4,img_5,img_6,img_7,img_8,...,domain_Naturaleza / animales,pred_Aguila,pred_Araña,pred_Elefante,pred_Mariposa,pred_Pajaro,pred_Pato,pred_Serpiente,pred_Tigre,pred_Tortuga
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## 4. Configurar pesos de similitud

Estos pesos controlan cuánto influye cada bloque en la recuperación.  
He dejado un peso alto al bloque de **objetivos XAI**, porque aquí sí queremos que entre en la similitud.


In [21]:

WEIGHTS = {
    "profile": 0.30,
    "preferences": 0.20,
    "xai_goals": 0.30,
    "image": 0.20,

}

print("Pesos actuales:", WEIGHTS)
print("Suma:", sum(WEIGHTS.values()))


Pesos actuales: {'profile': 0.3, 'preferences': 0.2, 'xai_goals': 0.3, 'image': 0.2}
Suma: 1.0


## 4.b Similitud visual con SSIM

Para el bloque de imagen se calcula una matriz de similitud visual entre las imágenes originales mediante SSIM. El resto de bloques sigue usando similitud coseno sobre atributos tabulares.

In [22]:
SSIM_IMAGE_SIZE = (224, 224)


def build_image_path_by_id(image_metadata_path, image_map_path=None, descriptions_path=None):
    if Path(image_metadata_path).exists():
        merged = pd.read_csv(image_metadata_path).rename(
            columns={
                "original_image_path": "image_path",
                "model_predicted_class": "class_label",
            }
        )
        if "description_case_id" not in merged.columns:
            merged["description_case_id"] = pd.NA
    elif image_map_path is not None and descriptions_path is not None:
        image_map = pd.read_csv(image_map_path)
        descriptions = pd.read_csv(descriptions_path)
        original_rows = descriptions[descriptions["method"] == "original"][["case_id", "image_path"]]
        merged = image_map.merge(
            original_rows,
            left_on="description_case_id",
            right_on="case_id",
            how="left",
        )
    else:
        raise FileNotFoundError(f"No encuentro metadatos de imagen en {image_metadata_path}")
    path_by_id = {}
    for row in merged.itertuples(index=False):
        if pd.notna(row.image_path) and Path(row.image_path).exists():
            path_by_id[int(row.image_id)] = Path(row.image_path)
    return path_by_id, merged


def load_image_for_ssim(path, size=SSIM_IMAGE_SIZE):
    arr = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0
    gray = rgb2gray(arr)
    return resize(gray, size, anti_aliasing=True, preserve_range=True).astype(float)


def compute_image_ssim_by_id(image_path_by_id):
    image_ids = sorted(image_path_by_id)
    images = {image_id: load_image_for_ssim(path) for image_id, path in image_path_by_id.items()}
    matrix = pd.DataFrame(np.eye(len(image_ids)), index=image_ids, columns=image_ids, dtype=float)
    for i, image_id_a in enumerate(image_ids):
        for image_id_b in image_ids[i + 1:]:
            raw_score = ssim(images[image_id_a], images[image_id_b], data_range=1.0)
            score = float(np.clip(raw_score, 0.0, 1.0))
            matrix.loc[image_id_a, image_id_b] = score
            matrix.loc[image_id_b, image_id_a] = score
    return matrix


def image_similarity_for_ids(image_id_a, image_id_b, image_ssim_by_id):
    try:
        a = int(image_id_a)
        b = int(image_id_b)
    except Exception:
        return 0.0
    if a in image_ssim_by_id.index and b in image_ssim_by_id.columns:
        return float(image_ssim_by_id.loc[a, b])
    return 1.0 if a == b else 0.0


def build_case_image_ssim_matrix(df_cases, image_ssim_by_id):
    image_ids = df_cases["image_id"].values
    matrix = np.zeros((len(df_cases), len(df_cases)), dtype=float)
    for i, image_id_a in enumerate(image_ids):
        for j, image_id_b in enumerate(image_ids):
            matrix[i, j] = image_similarity_for_ids(image_id_a, image_id_b, image_ssim_by_id)
    np.fill_diagonal(matrix, 1.0)
    return matrix


image_path_by_id, image_map_merged = build_image_path_by_id(IMAGE_METADATA_PATH, IMAGE_MAP_PATH, DESCRIPTIONS_PATH)
print("Imágenes con ruta válida para SSIM:", len(image_path_by_id))
display(image_map_merged[["image_id", "class_label", "description_case_id", "image_path"]])

image_ssim_by_id = compute_image_ssim_by_id(image_path_by_id)
case_image_ssim = build_case_image_ssim_matrix(df, image_ssim_by_id)

print("Matriz SSIM por image_id:")
display(image_ssim_by_id.round(3))


Imágenes con ruta válida para SSIM: 11


,image_id,class_label,description_case_id,image_path
0,1,Aguila,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
1,2,Pato,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
2,3,Serpiente,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
3,4,Araña,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
4,5,Mariposa,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
5,6,Tigre,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
6,7,Pajaro,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
7,8,Mariposa,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
8,9,Aguila,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...
9,10,Elefante,<NA>,/Users/haojie/Desktop/TFM/Imagenes/original/00...


Matriz SSIM por image_id:


,1,2,3,4,5,6,7,8,9,10,11
1,1.000,0.143,0.141,0.212,0.112,0.068,0.132,0.097,0.186,0.091,0.080
2,0.143,1.000,0.327,0.478,0.259,0.081,0.312,0.135,0.595,0.198,0.147
3,0.141,0.327,1.000,0.539,0.267,0.120,0.381,0.205,0.498,0.201,0.216
4,0.212,0.478,0.539,1.000,0.350,0.137,0.495,0.251,0.706,0.289,0.255
5,0.112,0.259,0.267,0.350,1.000,0.085,0.273,0.160,0.335,0.158,0.153
6,0.068,0.081,0.120,0.137,0.085,1.000,0.114,0.092,0.115,0.069,0.091
7,0.132,0.312,0.381,0.495,0.273,0.114,1.000,0.213,0.459,0.215,0.236
8,0.097,0.135,0.205,0.251,0.160,0.092,0.213,1.000,0.200,0.129,0.168
9,0.186,0.595,0.498,0.706,0.335,0.115,0.459,0.200,1.000,0.251,0.202
10,0.091,0.198,0.201,0.289,0.158,0.069,0.215,0.129,0.251,1.000,0.130


## 5. Similitud entre todos los casos de la base

Se calcula una **similitud por bloque** y luego se hace una **media ponderada**. Los bloques tabulares usan coseno; el bloque de imagen puede usar SSIM si la matriz visual está disponible.


In [23]:
def compute_similarity_matrices(blocks, weights, precomputed_sims=None):
    precomputed_sims = precomputed_sims or {}
    sim_by_block = {}
    for block_name, block_df in blocks.items():
        if block_name in precomputed_sims:
            sim_by_block[block_name] = precomputed_sims[block_name]
        else:
            sim_by_block[block_name] = safe_cosine_matrix(block_df.values)

    total = np.zeros_like(next(iter(sim_by_block.values())))
    total_weight = 0.0
    for block_name, sim in sim_by_block.items():
        w = weights.get(block_name, 0.0)
        total += w * sim
        total_weight += w

    total = total / total_weight if total_weight > 0 else total
    np.fill_diagonal(total, 1.0)
    return sim_by_block, total

precomputed_sims = {"image": case_image_ssim} if "case_image_ssim" in globals() else None
sim_by_block, sim_total = compute_similarity_matrices(blocks, WEIGHTS, precomputed_sims=precomputed_sims)
sim_total.shape


(297, 297)

## 6. Funciones de recuperación

Hay dos modos:

1. **Buscar vecinos de un caso existente** (`case_id`)
2. **Buscar vecinos de una query nueva** (`query_dict`)


In [24]:

def blocks_for_query(query_dict, template_blocks):
    qdf = pd.DataFrame([query_dict])
    qblocks = prepare_case_dataframe(qdf)
    aligned = {}
    for block_name, qblock in qblocks.items():
        aligned[block_name] = align_columns(qblock, template_blocks[block_name].columns)
    return aligned

def image_ssim_query_vs_base(query_dict, base_df, image_ssim_by_id):
    query_image_id = query_dict.get("image_id")
    return np.array([
        image_similarity_for_ids(query_image_id, image_id, image_ssim_by_id)
        for image_id in base_df["image_id"].values
    ], dtype=float)


def similarity_query_vs_base(
    query_blocks,
    base_blocks,
    weights,
    query_dict=None,
    base_df=None,
    image_ssim_by_id=None,
):
    sims = {}
    for block_name in base_blocks:
        if (
            block_name == "image"
            and query_dict is not None
            and base_df is not None
            and image_ssim_by_id is not None
        ):
            sims[block_name] = image_ssim_query_vs_base(query_dict, base_df, image_ssim_by_id)
        else:
            sims[block_name] = safe_cosine_query_vs_base(
                query_blocks[block_name].values,
                base_blocks[block_name].values
            )
    total = np.zeros(base_blocks["profile"].shape[0], dtype=float)
    total_weight = 0.0
    for block_name, sim in sims.items():
        w = weights.get(block_name, 0.0)
        total += w * sim
        total_weight += w
    total = total / total_weight if total_weight > 0 else total
    return sims, total

def get_neighbors_from_existing_case(
    df,
    sim_total,
    case_id,
    k=5,
    same_image_only=False,
    exclude_same_user=False
):
    if case_id not in set(df["case_id"]):
        raise ValueError(f"case_id no encontrado: {case_id}")

    idx = df.index[df["case_id"] == case_id][0]
    scores = sim_total[idx].copy()

    candidates = df.copy()
    candidates["similarity"] = scores

    # quitar el propio caso
    candidates = candidates[candidates["case_id"] != case_id]

    if same_image_only:
        img = df.loc[idx, "image_id"]
        candidates = candidates[candidates["image_id"] == img]

    if exclude_same_user:
        usr = df.loc[idx, "user_id"]
        candidates = candidates[candidates["user_id"] != usr]

    candidates = candidates.sort_values("similarity", ascending=False).head(k)
    return candidates

def get_neighbors_from_query(
    df,
    base_blocks,
    weights,
    query_dict,
    k=5,
    same_image_only=False,
    image_ssim_by_id=None,
):
    query_blocks = blocks_for_query(query_dict, base_blocks)
    _, total_sim = similarity_query_vs_base(
        query_blocks,
        base_blocks,
        weights,
        query_dict=query_dict,
        base_df=df,
        image_ssim_by_id=image_ssim_by_id,
    )

    candidates = df.copy()
    candidates["similarity"] = total_sim

    if same_image_only and "image_id" in query_dict:
        candidates = candidates[candidates["image_id"] == query_dict["image_id"]]

    candidates = candidates.sort_values("similarity", ascending=False).head(k)
    return candidates


## 7. Ejemplo 1: vecinos de un caso ya existente

In [25]:

example_case_id = df["case_id"].iloc[0]
neighbors_existing = get_neighbors_from_existing_case(
    df=df,
    sim_total=sim_total,
    case_id=example_case_id,
    k=5,
    same_image_only=True,
    exclude_same_user=True,
)

print("Caso consulta:", example_case_id)
display(
    df.loc[df["case_id"] == example_case_id, [
        "case_id", "user_id", "image_id", "selected_option",
        "ai_knowledge_level", "domain_knowledge_level",
        "preferred_response_length", "preferred_technical_level",
        "main_goals_raw"
    ]]
)
print("\nVecinos encontrados:")
display(
    neighbors_existing[[
        "case_id", "user_id", "image_id", "selected_option",
        "similarity", "mean_helpfulness", "satisfaction", "confidence", "understanding",
        "main_goals_raw"
    ]]
)


Caso consulta: C0001


,case_id,user_id,image_id,selected_option,ai_knowledge_level,domain_knowledge_level,preferred_response_length,preferred_technical_level,main_goals_raw
0,C0001,U001,1,Opción B,4,3,Media: explicación breve con algo de detalle,"Intermedio: algunos términos técnicos, pero fá...",Confianza: sentir mayor seguridad en la respue...



Vecinos encontrados:


,case_id,user_id,image_id,selected_option,similarity,mean_helpfulness,satisfaction,confidence,understanding,main_goals_raw
275,C0276,U026,1,Opción A,0.861226,2.333333,2,3,2,Transparencia: entender por qué el sistema tom...
286,C0287,U027,1,Opción B,0.798629,5.000000,5,5,5,Transparencia: entender por qué el sistema tom...
264,C0265,U025,1,Opción B,0.796533,3.000000,3,3,3,Transparencia: entender por qué el sistema tom...
165,C0166,U016,1,Opción A,0.783702,2.000000,2,2,2,Confianza: sentir mayor seguridad en la respue...
154,C0155,U015,1,Opción B,0.768317,4.333333,4,4,5,Transparencia: entender por qué el sistema tom...


## 8. Ejemplo 2: vecinos de una query nueva

Aquí no buscamos por `case_id`, sino por una query manual.  
Fíjate en que **`main_goals_raw` entra en la query**.


In [26]:

query_dict = {
    "image_id": 3,
    "age_range": "25–34",
    "education_level": "Master",
    "occupation_raw": "Investigador/a (académico)",
    "ai_knowledge_level": 4,
    "domain_knowledge_level": 3,
    "preferred_response_length": "Media: explicación breve con algo de detalle",
    "preferred_technical_level": "Intermedio: algunos términos técnicos, pero fáciles de seguir",
    "preferred_format": "Texto breve + imagen",
    "preferred_explanation_types_raw": "Explicaciones con ejemplos similares, Explicaciones visuales (zonas destacadas en la imagen)",
    "main_goals_raw": "Transparencia: entender por qué el sistema tomó esa decisión, Confianza: sentir mayor seguridad en la respuesta del sistema, Eficiencia: comprender la respuesta de forma rápida",
    "perceived_error_impact": "Medio: el error podría causar cierta confusión o problema",
}

neighbors_query = get_neighbors_from_query(
    df=df,
    base_blocks=blocks,
    weights=WEIGHTS,
    query_dict=query_dict,
    k=5,
    same_image_only=False,  # False permite que SSIM compare imágenes distintas
    image_ssim_by_id=image_ssim_by_id,
)

display(neighbors_query[[
    "case_id", "user_id", "image_id", "selected_option",
    "similarity", "mean_helpfulness", "satisfaction", "confidence", "understanding",
    "main_goals_raw"
]])


,case_id,user_id,image_id,selected_option,similarity,mean_helpfulness,satisfaction,confidence,understanding,main_goals_raw
288,C0289,U027,3,Opción B,0.718875,5.000000,5,5,5,Transparencia: entender por qué el sistema tom...
222,C0223,U021,3,Opción A,0.713968,3.333333,3,4,3,Transparencia: entender por qué el sistema tom...
90,C0091,U009,3,Opción B,0.705679,3.000000,3,3,3,Eficiencia: comprender la respuesta de forma r...
46,C0047,U005,3,Opción E,0.658944,4.666667,5,5,4,Transparencia: entender por qué el sistema tom...
35,C0036,U004,3,Opción A,0.632710,4.333333,4,4,5,Transparencia: entender por qué el sistema tom...


## 9. Recomendar explicación a partir de los vecinos

Se calcula una puntuación por opción de explicación usando:
- similitud del vecino
- utilidad media (`mean_helpfulness`)
- satisfacción
- confianza
- comprensión


In [27]:

def recommend_explanation(neighbors_df):
    tmp = neighbors_df.copy()
    for col in ["mean_helpfulness", "satisfaction", "confidence", "understanding", "similarity"]:
        tmp[col] = pd.to_numeric(tmp[col], errors="coerce").fillna(0)

    tmp["utility_score"] = (
        0.40 * tmp["mean_helpfulness"] +
        0.20 * tmp["satisfaction"] +
        0.20 * tmp["confidence"] +
        0.20 * tmp["understanding"]
    )
    tmp["weighted_vote"] = tmp["utility_score"] * tmp["similarity"]

    ranking = (
        tmp.groupby("selected_option", dropna=False)
        .agg(
            n_neighbors=("case_id", "count"),
            mean_similarity=("similarity", "mean"),
            mean_utility=("utility_score", "mean"),
            total_weighted_vote=("weighted_vote", "sum"),
        )
        .sort_values(["total_weighted_vote", "mean_similarity"], ascending=False)
        .reset_index()
    )
    return ranking

ranking_existing = recommend_explanation(neighbors_existing)
ranking_query = recommend_explanation(neighbors_query)

print("Recomendación a partir de vecinos del caso existente:")
display(ranking_existing)

print("Recomendación a partir de vecinos de la query nueva:")
display(ranking_query)


Recomendación a partir de vecinos del caso existente:


,selected_option,n_neighbors,mean_similarity,mean_utility,total_weighted_vote
0,Opción B,3,0.787826,4.111111,9.712116
1,Opción A,2,0.822464,2.166667,3.576932


Recomendación a partir de vecinos de la query nueva:


,selected_option,n_neighbors,mean_similarity,mean_utility,total_weighted_vote
0,Opción B,2,0.712277,4.000000,5.711414
1,Opción A,2,0.673339,3.833333,5.121634
2,Opción E,1,0.658944,4.666667,3.075071


## 9.b Ejemplos de vecinos similares

Este bloque recupera, para cada opción recomendada, un vecino representativo y genera una explicación breve de por qué se parece al caso de referencia o a la query.


In [28]:
SIMILARITY_REASON_FIELDS = [
    "age_range",
    "education_level",
    "ai_knowledge_level",
    "domain_knowledge_level",
    "preferred_response_length",
    "preferred_technical_level",
    "preferred_format",
]

SIMILARITY_REASON_LABELS = {
    "age_range": "rango de edad",
    "education_level": "nivel educativo",
    "ai_knowledge_level": "conocimiento de IA",
    "domain_knowledge_level": "conocimiento del dominio",
    "preferred_response_length": "longitud de respuesta preferida",
    "preferred_technical_level": "nivel técnico preferido",
    "preferred_format": "formato preferido",
}

def build_similarity_reason(example_row, reference_row):
    matches = []
    for field in SIMILARITY_REASON_FIELDS:
        if field not in example_row.index or field not in reference_row.index:
            continue
        example_value = example_row[field]
        reference_value = reference_row[field]
        if pd.notna(example_value) and pd.notna(reference_value) and example_value == reference_value:
            matches.append(SIMILARITY_REASON_LABELS.get(field, field))

    if matches:
        return "Similar porque coincide en: " + ", ".join(matches[:3])
    return "Similar por el perfil global y la puntuación de similitud."

def explain_recommendation_examples(neighbors_df, reference_row):
    tmp = neighbors_df.copy()
    for col in ["mean_helpfulness", "satisfaction", "confidence", "understanding", "similarity"]:
        tmp[col] = pd.to_numeric(tmp[col], errors="coerce").fillna(0)

    tmp["utility_score"] = (
        0.40 * tmp["mean_helpfulness"] +
        0.20 * tmp["satisfaction"] +
        0.20 * tmp["confidence"] +
        0.20 * tmp["understanding"]
    )
    tmp["weighted_vote"] = tmp["utility_score"] * tmp["similarity"]

    idx_best_example = tmp.groupby("selected_option", dropna=False)["weighted_vote"].idxmax()
    examples = tmp.loc[idx_best_example].copy()
    examples["similarity_reason"] = examples.apply(
        lambda row: build_similarity_reason(row, reference_row),
        axis=1,
    )

    return (
        examples[[
            "selected_option",
            "case_id",
            "image_id",
            "similarity",
            "weighted_vote",
            "mean_helpfulness",
            "satisfaction",
            "confidence",
            "understanding",
            "similarity_reason",
        ]]
        .sort_values(["weighted_vote", "similarity"], ascending=False)
        .rename(columns={
            "case_id": "example_case_id",
            "image_id": "example_image_id",
            "similarity": "example_similarity",
        })
        .reset_index(drop=True)
    )

reference_existing = df.loc[df["case_id"] == example_case_id].iloc[0]
reference_query = pd.Series(query_dict)

examples_existing = explain_recommendation_examples(neighbors_existing, reference_existing)
examples_query = explain_recommendation_examples(neighbors_query, reference_query)

print("Ejemplo de vecino similar por opción para el caso existente:")
display(examples_existing)

print("Ejemplo de vecino similar por opción para la query nueva:")
display(examples_query)


Ejemplo de vecino similar por opción para el caso existente:


,selected_option,example_case_id,example_image_id,example_similarity,weighted_vote,mean_helpfulness,satisfaction,confidence,understanding,similarity_reason
0,Opción B,C0287,1,0.798629,3.993143,5.000000,5,5,5,"Similar porque coincide en: rango de edad, niv..."
1,Opción A,C0276,1,0.861226,2.009528,2.333333,2,3,2,"Similar porque coincide en: rango de edad, niv..."


Ejemplo de vecino similar por opción para la query nueva:


,selected_option,example_case_id,example_image_id,example_similarity,weighted_vote,mean_helpfulness,satisfaction,confidence,understanding,similarity_reason
0,Opción B,C0289,3,0.718875,3.594377,5.000000,5,5,5,Similar porque coincide en: conocimiento de IA...
1,Opción E,C0047,3,0.658944,3.075071,4.666667,5,5,4,"Similar porque coincide en: nivel educativo, c..."
2,Opción A,C0036,3,0.632710,2.741742,4.333333,4,4,5,Similar porque coincide en: conocimiento de IA...


## 10. Exportar salidas

Se guardan:
- matriz de atributos numéricos por bloque
- top vecinos por caso
- ranking de recomendaciones para una query de ejemplo


In [29]:

# Exportar bloques numéricos
for block_name, block_df in blocks.items():
    out_path = OUTPUT_DIR / f"block_{block_name}.csv"
    block_df.to_csv(out_path, index=False)

# Exportar top-5 vecinos para cada caso (mismo image_id y distinto user_id)
all_neighbors = []
for cid in df["case_id"]:
    topn = get_neighbors_from_existing_case(
        df=df,
        sim_total=sim_total,
        case_id=cid,
        k=5,
        same_image_only=True,
        exclude_same_user=True,
    ).copy()
    topn.insert(0, "query_case_id", cid)
    all_neighbors.append(topn)

neighbors_all_df = pd.concat(all_neighbors, ignore_index=True)
neighbors_all_path = OUTPUT_DIR / "top5_neighbors_same_image_other_user.csv"
neighbors_all_df.to_csv(neighbors_all_path, index=False)


# Exportar matriz SSIM entre imágenes
if "image_ssim_by_id" in globals():
    image_ssim_by_id.to_csv(OUTPUT_DIR / "image_ssim_by_id.csv")

# Exportar rankings ejemplo
ranking_existing.to_csv(OUTPUT_DIR / "ranking_existing_case_example.csv", index=False)
ranking_query.to_csv(OUTPUT_DIR / "ranking_new_query_example.csv", index=False)

print("Salidas guardadas en:", OUTPUT_DIR)
print(sorted([p.name for p in OUTPUT_DIR.iterdir()]))


Salidas guardadas en: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/base_de_casos/cbr_similarity_outputs
['block_image.csv', 'block_preferences.csv', 'block_profile.csv', 'block_xai_goals.csv', 'cbr_loo_clean_folds.csv', 'cbr_loo_clean_k_sensitivity.csv', 'cbr_loo_clean_k_sensitivity_with_baseline_f1.csv', 'cbr_loo_clean_metrics_by_field.csv', 'cbr_loo_clean_metrics_by_field_k11_with_baseline_f1.csv', 'cbr_loo_clean_neighbors.csv', 'cbr_loo_clean_predictions.csv', 'cbr_loo_clean_summary.csv', 'cbr_loo_clean_summary_k11_with_baseline_f1.csv', 'image_ssim_by_id.csv', 'ranking_existing_case_example.csv', 'ranking_new_query_example.csv', 'top5_neighbors_same_image_other_user.csv']


## 11.

- `WEIGHTS`: cuánto pesa cada bloque en la recuperación.
- `same_image_only=True`: si quieres buscar solo dentro de la misma imagen o no.
- `exclude_same_user=True`: útil para no comparar un usuario consigo mismo.
- `recommend_explanation()`: si quieres dar más peso a satisfacción, confianza o comprensión.
- `prepare_case_dataframe()`: si luego añades metadatos de imagen o embeddings, este es el sitio donde integrarlos.
